# ✏️ Draw & Guess — Sketch Classifier v2
Enhanced version with 30 categories, 10k samples per class, ResNet skip connections, and a smarter Gradio app.

**Run everything:** Runtime → Run all (`Ctrl+F9`) — enable GPU first!

| Step | What happens |
|------|---------------|
| 1 | Install & import |
| 2 | Download QuickDraw (30 categories × 10k) |
| 3 | Explore — samples + mean images |
| 4 | Preprocess & tf.data pipeline |
| 5 | Build ResNet-style model |
| 6 | Train |
| 7 | Evaluate — curves, per-class accuracy, confusion matrix, mistakes |
| 8 | Save |
| 9 | Draw & guess live (Gradio, with confidence threshold) |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio seaborn pillow --quiet

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D, Flatten,
    Dense, Dropout, BatchNormalization, Add, Input, Activation
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print('✅ All libraries imported!')
print(f'   TensorFlow: {tf.__version__}')

## Step 2 — Download QuickDraw Dataset (30 categories)

In [ ]:
# 30 categories — a good mix of animals, objects, vehicles, food, nature
CATEGORIES = [
    'cat', 'dog', 'car', 'house', 'tree',
    'sun', 'fish', 'bird', 'bicycle', 'airplane',
    'pizza', 'umbrella', 'clock', 'star', 'apple',
    'flower', 'mountain', 'boat', 'guitar', 'hat',
    'lion', 'elephant', 'butterfly', 'mushroom', 'rainbow',
    'truck', 'cake', 'moon', 'eye', 'spider'
]

NUM_CLASSES      = len(CATEGORIES)
SAMPLES_PER_CLASS = 10000   # was 5000 — doubled for better accuracy
BASE_URL         = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'

X_list, y_list = [], []

for idx, category in enumerate(CATEGORIES):
    url  = BASE_URL + category.replace(' ', '%20') + '.npy'
    path = f'{category}.npy'

    if not os.path.exists(path):
        os.system(f'wget -q "{url}" -O "{path}"')

    data = np.load(path)[:SAMPLES_PER_CLASS]
    X_list.append(data)
    y_list.append(np.full(len(data), idx))
    print(f'  [{idx+1:2d}/{NUM_CLASSES}] {category:<12} — {len(data):,} drawings loaded')

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

print(f'\n✅ Dataset ready!')
print(f'   Total drawings : {len(X):,}')
print(f'   Classes        : {NUM_CLASSES}')

## Step 3 — Explore & Visualize

In [ ]:
# --- 3a: Sample drawings grid ---
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(6, NUM_CLASSES * 2))
for row, category in enumerate(CATEGORIES):
    idxs = np.random.choice(np.where(y == row)[0], 3, replace=False)
    for col, i in enumerate(idxs):
        axes[row, col].imshow(X[i].reshape(28, 28), cmap='gray_r')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(category, fontsize=8, loc='left')
plt.suptitle('3 samples per category', y=1.002, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3b: Mean image per category ---
# Average all drawings in each class — shows what the model's 'ideal' version looks like
X_reshaped = X.reshape(-1, 28, 28).astype('float32')

cols = 6
rows = (NUM_CLASSES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2))
axes = axes.flatten()

for i, category in enumerate(CATEGORIES):
    mean_img = X_reshaped[y == i].mean(axis=0)
    axes[i].imshow(mean_img, cmap='gray_r')
    axes[i].set_title(category, fontsize=9)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Mean drawing per category\n(average of 10,000 samples)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('💡 Blurry = high variation in how people draw it (harder to classify)')
print('   Sharp  = people draw it consistently (easier to classify)')

## Step 4 — Preprocess & Build tf.data Pipeline

In [ ]:
BATCH_SIZE = 128
AUTOTUNE   = tf.data.AUTOTUNE

# Reshape and normalize
X = X.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1,  seed=SEED),
    tf.keras.layers.RandomZoom(0.1,      seed=SEED),
    tf.keras.layers.RandomTranslation(0.1, 0.1, seed=SEED),
])

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

print('✅ Preprocessing complete!')
print(f'   Train   : {len(X_train):,} samples')
print(f'   Val     : {len(X_val):,} samples')
print(f'   Test    : {len(X_test):,} samples')

## Step 5 — Build the ResNet-Style Model
> **What is a residual block?** A skip connection adds the block's input directly to its output: `output = F(x) + x`. This lets gradients flow backward through the shortcut, solving the vanishing gradient problem that makes deep networks hard to train.

In [ ]:
def residual_block(x, filters, downsample=False):
    stride = 2 if downsample else 1
    shortcut = x

    # Main path
    x = Conv2D(filters, (3, 3), strides=stride, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), strides=1, padding='same')(x)
    x = BatchNormalization()(x)

    # Shortcut path — match dimensions if downsampling or filter size changed
    if downsample or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add skip connection
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x


# Build model using Keras Functional API
inputs = Input(shape=(28, 28, 1))

# Entry block
x = Conv2D(32, (3, 3), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

# Residual blocks
x = residual_block(x, 32)
x = residual_block(x, 64,  downsample=True)
x = residual_block(x, 64)
x = residual_block(x, 128, downsample=True)
x = residual_block(x, 128)

# Classification head
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total parameters : {model.count_params():,}')
model.summary()

## Step 6 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=2, min_lr=1e-6, verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\n✅ Training complete — stopped at epoch {len(history.history["loss"])}')

## Step 7 — Evaluate the Model

In [ ]:
loss, acc = model.evaluate(test_ds, verbose=0)
print(f'Test Accuracy : {acc*100:.2f}%')
print(f'Test Loss     : {loss:.4f}')
print(f'Chance level  : {100/NUM_CLASSES:.1f}%  (random guessing)\n')

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history.history['loss'],         label='Train')
axes[1].plot(history.history['val_loss'],     label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predictions on test set
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

In [ ]:
# --- Per-class accuracy bar chart ---
# Much easier to read than a 30x30 confusion matrix
cm = confusion_matrix(y_test, y_pred)
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
mean_acc = per_class_acc.mean()

# Color bars by accuracy: green = good, red = struggling
colors = ['#2ecc71' if a >= mean_acc else '#e74c3c' for a in per_class_acc]

plt.figure(figsize=(14, 5))
bars = plt.bar(CATEGORIES, per_class_acc, color=colors, edgecolor='white')
plt.axhline(y=mean_acc, color='gray', linestyle='--', linewidth=1.2,
            label=f'Mean: {mean_acc:.1f}%')
plt.title('Accuracy per category')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

# Print best and worst
ranked = sorted(zip(per_class_acc, CATEGORIES), reverse=True)
print('Top 5 easiest categories:')
for a, c in ranked[:5]:
    print(f'  {c:<12} {a:.1f}%')
print('\nTop 5 hardest categories:')
for a, c in ranked[-5:]:
    print(f'  {c:<12} {a:.1f}%')

In [ ]:
# --- Confusion matrix (30x30) ---
plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualize mistakes ---
wrong_idx = np.where(y_pred != y_test)[0]
print(f'Total mistakes: {len(wrong_idx)} / {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')

plt.figure(figsize=(15, 8))
for i, idx in enumerate(wrong_idx[:20]):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray_r')
    true_label = CATEGORIES[int(y_test[idx])]
    pred_label = CATEGORIES[int(y_pred[idx])]
    plt.title(f'True: {true_label}\nPred: {pred_label}', fontsize=7, color='crimson')
    plt.axis('off')
plt.suptitle('Sketches the model got wrong', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Learning rate history ---
lr_history = history.history.get('learning_rate', history.history.get('lr', []))
if lr_history:
    plt.figure(figsize=(8, 3))
    plt.plot(lr_history, marker='o', color='darkorange')
    plt.title('Learning rate over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Learning rate')
    plt.yscale('log')
    plt.tight_layout()
    plt.show()
    drops = [i for i in range(1, len(lr_history)) if lr_history[i] < lr_history[i-1]]
    print(f'LR reduced at epochs: {[d+1 for d in drops]}' if drops else 'LR was never reduced.')

## Step 8 — Save the Model

In [ ]:
model.save('sketch_classifier_v2.keras')
print('✅ Model saved as sketch_classifier_v2.keras')
# To reload: model = load_model('sketch_classifier_v2.keras')

## Step 9 — Draw & Guess Live (Gradio)
> Draw any of the 30 categories. Shows top 3 guesses with confidence. If the model is unsure (< 40% confidence) it says so instead of guessing wrong.

In [ ]:
import gradio as gr

CONFIDENCE_THRESHOLD = 0.40

def predict_sketch(img):
    if img is None:
        return {c: 0.0 for c in CATEGORIES}

    # Handle both dict (newer Gradio) and array inputs
    if isinstance(img, dict):
        img = img.get('composite', img.get('background', None))
    if img is None:
        return {c: 0.0 for c in CATEGORIES}

    # Resize to 28x28 grayscale
    pil_img = Image.fromarray(img.astype('uint8'))
    pil_img = pil_img.convert('L')
    pil_img = pil_img.resize((28, 28), Image.LANCZOS)

    arr = np.array(pil_img).astype('float32')

    # Invert: QuickDraw = white strokes on black, sketchpad = black strokes on white
    arr = 255.0 - arr
    arr = arr / 255.0
    arr = arr.reshape(1, 28, 28, 1)

    probs = model.predict(arr, verbose=0)[0]
    top_prob = float(np.max(probs))

    # Confidence threshold — avoid confident wrong answers
    if top_prob < CONFIDENCE_THRESHOLD:
        return {'Not sure — keep drawing!': 1.0}

    return {CATEGORIES[i]: float(probs[i]) for i in range(NUM_CLASSES)}


# Build category hint string
hint = ' · '.join(CATEGORIES)

with gr.Blocks() as app:
    gr.Markdown(f'# ✏️ Draw & Guess\n**Draw one of these 30 categories:**\n\n{hint}')
    with gr.Row():
        with gr.Column():
            sketchpad = gr.Sketchpad(
                label='Draw here',
                type='numpy'
            )
            clear_btn = gr.Button('Clear')
        with gr.Column():
            output = gr.Label(
                num_top_classes=3,
                label='Top 3 guesses'
            )
            gr.Markdown(
                f'*Model says "Not sure" when confidence is below {int(CONFIDENCE_THRESHOLD*100)}%.*'
            )

    sketchpad.change(fn=predict_sketch, inputs=sketchpad, outputs=output)
    clear_btn.click(fn=lambda: None, inputs=None, outputs=sketchpad)

app.launch(debug=False)